In [1]:
!pip install ultralytics
!pip install torch torchvision
!pip install pillow
!pip install pyyaml
!pip install opencv-python

In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="XR0oOiu6HNTti6EO74va")
project = rf.workspace("occluded-and-cluttered-scenes").project("annotation-vgeil")
version = project.version(11)
dataset = version.download("coco-mmdetection")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to annotation-11 in coco-mmdetection:: 100%|██████████| 7588/7588 [00:03<00:00, 1904.75it/s]


In [ ]:
import json
import os
from pathlib import Path
from PIL import Image
import shutil

def convert_coco_to_yolo(coco_json_path, images_dir, output_dir):
    """
    Convert COCO JSON format to YOLO format with explicit float casting
    """
    print(f"Converting {coco_json_path} to YOLO format...")

    with open(coco_json_path, 'r') as f:
        coco_data = json.load(f)

    output_images_dir = Path(output_dir) / "images"
    output_labels_dir = Path(output_dir) / "labels"
    output_images_dir.mkdir(parents=True, exist_ok=True)
    output_labels_dir.mkdir(parents=True, exist_ok=True)

    categories = {cat['id']: idx for idx, cat in enumerate(coco_data['categories'])}

    img_to_anns = {}
    for ann in coco_data['annotations']:
        img_id = ann['image_id']
        if img_id not in img_to_anns:
            img_to_anns[img_id] = []
        img_to_anns[img_id].append(ann)

    converted_count = 0
    for img_info in coco_data['images']:
        img_id = img_info['id']
        img_filename = img_info['file_name']
        
       
        img_width = float(img_info['width'])
        img_height = float(img_info['height'])

        src_img_path = Path(images_dir) / img_filename
        dst_img_path = output_images_dir / img_filename
        if src_img_path.exists():
            shutil.copy(src_img_path, dst_img_path)
        else:
            continue

        label_filename = Path(img_filename).stem + '.txt'
        label_path = output_labels_dir / label_filename

        yolo_annotations = []
        if img_id in img_to_anns:
            for ann in img_to_anns[img_id]:
                x, y, w, h = map(float, ann['bbox'])

                x_center = (x + w / 2) / img_width
                y_center = (y + h / 2) / img_height
                norm_width = w / img_width
                norm_height = h / img_height

                class_id = categories[ann['category_id']]
                yolo_annotations.append(f"{class_id} {x_center:.6f} {y_center:.6f} {norm_width:.6f} {norm_height:.6f}")

        with open(label_path, 'w') as f:
            f.write('\n'.join(yolo_annotations))

        converted_count += 1
        if converted_count % 500 == 0:
            print(f"Converted {converted_count} images...")

    print(f"✓ Conversion complete!")
    return [cat['name'] for cat in sorted(coco_data['categories'], key=lambda x: categories[x['id']])]

In [3]:
def create_dataset_yaml(output_path, train_dir, val_dir, test_dir, class_names):
    """
    Create the dataset.yaml file required by Ultralytics
    """
    yaml_content = f"""# Dataset configuration for RT-DETR training

path: .  # Root directory (leave as . if using absolute paths)
train: {train_dir}/images
val: {val_dir}/images
test: {test_dir}/images

# Number of classes
nc: {len(class_names)}

# Class names
names: {class_names}
"""

    with open(output_path, 'w') as f:
        f.write(yaml_content)

    print(f"✓ Dataset YAML created: {output_path}")
    return output_path

In [4]:
def setup_yolo_dataset(base_dir):
    """
    Convert COCO format dataset to YOLO format

    Args:
        base_dir: Base directory containing train/, valid/, test/ folders
                  Each should have images and _annotations.coco.json
    """
    base_path = Path(base_dir)
    output_base = base_path / "yolo_format"

    print("="*60)
    print("CONVERTING COCO DATASET TO YOLO FORMAT")
    print("="*60)

    # Convert train set
    train_json = base_path / "train" / "_annotations.coco.json"
    train_images = base_path / "train"
    train_output = output_base / "train"
    class_names = convert_coco_to_yolo(train_json, train_images, train_output)

    # Convert validation set
    val_json = base_path / "valid" / "_annotations.coco.json"
    val_images = base_path / "valid"
    val_output = output_base / "valid"
    convert_coco_to_yolo(val_json, val_images, val_output)

    # Convert test set
    test_json = base_path / "test" / "_annotations.coco.json"
    test_images = base_path / "test"
    test_output = output_base / "test"
    convert_coco_to_yolo(test_json, test_images, test_output)

    # Create dataset.yaml
    yaml_path = output_base / "dataset.yaml"
    create_dataset_yaml(
        yaml_path,
        str(train_output.absolute()),
        str(val_output.absolute()),
        str(test_output.absolute()),
        class_names
    )

    print("\n" + "="*60)
    print("DATASET CONVERSION COMPLETE!")
    print("="*60)
    print(f"Dataset YAML: {yaml_path}")
    print(f"Classes: {class_names}")

    return str(yaml_path)

In [5]:
from ultralytics import RTDETR
import torch

def train_rtdetr(dataset_yaml, epochs=50, batch_size=8, image_size=640, device=None):
    """
    Train RT-DETR model using Ultralytics (GPU recommended)
    """

    print("\n" + "="*60)
    print("STARTING RT-DETR TRAINING")
    print("="*60)

    # Auto-detect device
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

    if device == 'cuda':
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")
        # Aggressive GPU cache clearing
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        except Exception as e:
            print(f"Note: GPU memory management issue, proceeding anyway: {e}")
    else:
        print("Using CPU (slow)")

    print(f"Epochs: {epochs}")
    print(f"Batch size: {batch_size}")
    print(f"Image size: {image_size}")

    # Load pretrained RT-DETR
    model = RTDETR('rtdetr-l.pt')

    # Train
    results = model.train(
        data=dataset_yaml,
        epochs=epochs,
        batch=batch_size,
        imgsz=image_size,
        device=device,
        workers=0,  # No data loading workers to save memory
        patience=20,
        save=True,
        project='rtdetr_runs',
        name='train',
        exist_ok=True,
        pretrained=True,
        amp=True,  # Mixed precision to save memory
        cache=False,  # Disable caching to avoid OOM - load from disk instead
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=10,
        translate=0.1,
        scale=0.5,
        flipud=0.5,
        fliplr=0.5,

        # Optimizer
        optimizer='AdamW',
        lr0=1e-4,
        lrf=0.01,
        weight_decay=1e-4,
        warmup_epochs=3,

        # Loss weights
        box=5.0,
        cls=1.0,
        dfl=1.5,

        plots=True,
        verbose=True
    )

    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)
    print("Best model: runs/detect/rtdetr_runs/train/weights/best.pt")

    return model, results

In [6]:
def validate_rtdetr(model_path, dataset_yaml, device='cuda'):
    """
    Validate RT-DETR model on test set
    """
    print("\n" + "="*60)
    print("VALIDATING RT-DETR MODEL")
    print("="*60)

    # Load trained model
    model = RTDETR(model_path)

    # Validate
    results = model.val(
        data=dataset_yaml,
        device=device,
        batch=1,
        workers=0,
        split='test',
        plots=True,
        save_json=True,
        verbose=True
    )

    print("\n" + "="*60)
    print("VALIDATION RESULTS")
    print("="*60)
    print(f"mAP50: {results.box.map50:.4f}")
    print(f"mAP50-95: {results.box.map:.4f}")
    print(f"Precision: {results.box.mp:.4f}")
    print(f"Recall: {results.box.mr:.4f}")

    return results

In [7]:
def test_on_images(model_path, image_paths, save_dir='predictions', conf_threshold=0.25):
    """
    Test RT-DETR model on individual images
    """
    print("\n" + "="*60)
    print("TESTING ON IMAGES")
    print("="*60)

    # Load model
    model = RTDETR(model_path)

    # Predict
    results = model.predict(
        source=image_paths,
        conf=conf_threshold,
        save=True,
        project=save_dir,
        name='predictions',
        exist_ok=True,
        show_labels=True,
        show_conf=True,
        line_width=2
    )

    print(f"✓ Predictions saved to: {save_dir}/predictions/")

    # Print results for each image
    for i, result in enumerate(results):
        # print(f"\nImage {i+1}:")
        # print(f"  Detected {len(result.boxes)} objects")
        for box in result.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            # print(f"  - Class {cls}: {conf:.2f}")

    return results

In [ ]:
def main():
    """
    Main execution function - Complete pipeline
    """
    # ========================================
    # CONFIGURATION
    # ========================================

    # UPDATE THESE PATHS FOR YOUR DATASET
    BASE_DIR = r"annotation-11"

    # Training parameters
    EPOCHS = 100
    BATCH_SIZE = 16  # Batch size of 16 should fit in 16GB VRAM with image size 640x640
    IMAGE_SIZE = 640  # Resize images to 640x640
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Clear GPU memory before starting (with error handling)
    if DEVICE == 'cuda':
        try:
            torch.cuda.empty_cache()
        except Exception as e:
            print(f"Could not clear GPU cache: {e}")
            print("Proceeding anyway...")

    # ========================================
    # STEP 1: Convert Dataset
    # ========================================
    print("\n🔄 Step 1: Converting COCO dataset to YOLO format...")
    dataset_yaml = setup_yolo_dataset(BASE_DIR)
 
    # ========================================
    # STEP 2: Train Model
    # ========================================
    print("\nStep 2: Training RT-DETR model...")
    model, train_results = train_rtdetr(
        dataset_yaml=dataset_yaml,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,
        device=DEVICE
    )

    # ========================================
    # STEP 3: Validate Model
    # ========================================
    print("\nStep 3: Validating model...")
    best_model_path = r"runs\detect\rtdetr_runs\train\weights\best.pt"
    val_results = validate_rtdetr(best_model_path, dataset_yaml, DEVICE)

    # ========================================
    # STEP 4: Test on Sample Images
    # ========================================
    print("\nStep 4: Testing on sample images...")
    test_images = f"{BASE_DIR}/test"  # Ultralytics will find all images automatically
    test_results = test_on_images(best_model_path, test_images)

    print("\n" + "="*60)
    print("COMPLETE PIPELINE FINISHED SUCCESSFULLY!")
    print("="*60)
    print(f"\nBest model: runs/detect/rtdetr_runs/train/weights/best.pt")
    print(f"\nLast model: runs/detect/rtdetr_runs/train/weights/last.pt")
    print(f"\nResults: rtdetr_runs/train/")
    
if __name__ == "__main__":
    main()


🔄 Step 1: Converting COCO dataset to YOLO format...
CONVERTING COCO DATASET TO YOLO FORMAT
Converting annotation-11\train\_annotations.coco.json to YOLO format...
Converted 500 images...
Converted 1000 images...
Converted 1500 images...
Converted 2000 images...
Converted 2500 images...
Converted 3000 images...
Converted 3500 images...
Converted 4000 images...
Converted 4500 images...
Converted 5000 images...
Converted 5500 images...
Converted 6000 images...
Converted 6500 images...
✓ Conversion complete!
Converting annotation-11\valid\_annotations.coco.json to YOLO format...
Converted 500 images...
✓ Conversion complete!
Converting annotation-11\test\_annotations.coco.json to YOLO format...
✓ Conversion complete!
✓ Dataset YAML created: annotation-11\yolo_format\dataset.yaml

DATASET CONVERSION COMPLETE!
Dataset YAML: annotation-11\yolo_format\dataset.yaml
Classes: ['glue', 'marker', 'medicine', 'white bottle']

Step 2: Training RT-DETR model...

STARTING RT-DETR TRAINING
Using GPU: N

c:\Users\user\anaconda3\envs\rtdetr_env\lib\site-packages\torch\autograd\graph.py:829: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      12.2G      0.439      1.764     0.2848         16        640: 100% ━━━━━━━━━━━━ 422/422 1.1s/it 7:42<1.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 1.6it/s 11.1s0.5s
                   all        561       1741      0.144      0.477       0.16     0.0967

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/100      12.7G     0.2942      1.457     0.2065         69        640: 0% ──────────── 0/422  0.8s

c:\Users\user\anaconda3\envs\rtdetr_env\lib\site-packages\torch\autograd\graph.py:829: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      12.7G     0.3806      1.148      0.205         30        640: 100% ━━━━━━━━━━━━ 422/422 1.4it/s 5:09<0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 1.8it/s 10.0s.6s
                   all        561       1741      0.803      0.956      0.896      0.514

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/100      12.5G     0.5274     0.6036     0.2852        103        640: 0% ──────────── 0/422  1.0s

...

c:\Users\user\anaconda3\envs\rtdetr_env\lib\site-packages\torch\autograd\graph.py:829: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     68/100      12.8G     0.1175     0.1988    0.06642         15        640: 100% ━━━━━━━━━━━━ 422/422 1.4it/s 4:53<0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.5it/s 7.1s0.4s
                   all        561       1741      0.979      0.994       0.99      0.896

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     69/100      12.5G     0.1059     0.1898    0.06233         70        640: 0% ──────────── 0/422  0.8s

c:\Users\user\anaconda3\envs\rtdetr_env\lib\site-packages\torch\autograd\graph.py:829: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     69/100      12.5G     0.1174     0.1994    0.06569         17        640: 100% ━━━━━━━━━━━━ 422/422 1.4it/s 4:53<0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.6it/s 7.0s0.4s
                   all        561       1741      0.981      0.994       0.99      0.899
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 49, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

69 epochs completed in 6.024 hours.
Optimizer stripped from C:\Ivander\rl_grasping_project\scripts\runs\detect\rtdetr_runs\train\weights\last.pt, 66.2MB
Optimizer stripped from C:\Ivander\rl_grasping_project\scripts\runs\detect\rtdetr_runs\train\weights\best.pt, 66.2MB

Validating C:\Ivander\rl_grasping_project\scripts\runs\detect\rtdetr_runs\train\weights\best.pt...
Ultralyt

: 